## Librerías

In [1]:
#%pip install requests pandas

## Llamada al API (con paginación) y carga al csv

In [3]:
import requests, pandas as pd, time

rows, url = [], "https://rickandmortyapi.com/api/character"

while url:
    for intento in range(3):  # hasta 3 reintentos por página
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and r.text.strip():
            data = r.json()
            break
        print(f"Fallo (status {r.status_code}), reintento {intento + 1}...")
        time.sleep(1)
    else:
        raise RuntimeError(f"No se pudo obtener {url} tras 3 intentos")

    for c in data["results"]:
        episode_ids = [int(ep.split("/")[-1]) for ep in c["episode"]]
        rows.append({
            "id": c["id"], "name": c["name"], "status": c["status"],
            "species": c["species"], "type": c["type"] if c["type"] else "Not specified", "gender": c["gender"],
            "origin": c["origin"]["name"], "location": c["location"]["name"],
            "n_episodes": len(c["episode"]), "episode_ids": ",".join(map(str, episode_ids)), "image": c["image"],
        })
    url = data["info"]["next"]

pd.DataFrame(rows).to_csv("RMcharacters.csv", index=False)